In [ ]:
# ==============================================================================
# ANÁLISE EPIDEMIOLÓGICA: PACIENTES ÚNICOS (GRÁFICO + TABELA EXATA)
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
import warnings

warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. CARREGAMENTO DOS DADOS (BASE PRINCIPAL)
# ------------------------------------------------------------------------------
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: O arquivo {file_path} não foi encontrado no diretório atual.")
    raise

# ------------------------------------------------------------------------------
# 2. RASTREAMENTO CLÍNICO (TEXT MINING + CHECKBOXES)
# ------------------------------------------------------------------------------
text_cols = [c for c in df.columns if df[c].dtype == object]
df_text = df[text_cols].fillna('')

# UDVI (Usuários de Drogas Injetáveis)
regex_injet = r'\b(injet[aá]ve[li]s?|udvi)\b'
cond_injet_text = df_text.apply(lambda x: x.astype(str).str.contains(regex_injet, case=False, regex=True)).any(axis=1)

cols_drogas = [c for c in df.columns if 'droga' in c.lower() or 'injet' in c.lower() or 'comportamento' in c.lower()]
cond_injet_str = pd.Series(False, index=df.index)
for c in cols_drogas:
    if 'injet' in c.lower() and 'choice=' in c.lower():
        is_checked = df[c].astype(str).str.lower().str.strip().isin(['checked', '1', '1.0', 'sim', 'verdadeiro'])
        cond_injet_str = cond_injet_str | is_checked

df['Uso_Injetaveis'] = (cond_injet_text | cond_injet_str).astype(int)

# Doenças Crônicas
regex_has = r'\b(hipertens[aã]o|has|press[aã]o alta)\b'
regex_dm = r'\b(diabetes|dm2|dm1|dm)\b'

cond_has_text = df_text.apply(lambda x: x.astype(str).str.contains(regex_has, case=False, regex=True)).any(axis=1)
cond_dm_text = df_text.apply(lambda x: x.astype(str).str.contains(regex_dm, case=False, regex=True)).any(axis=1)

cols_cronicas = [c for c in df.columns if 'doenças crônicas' in c.lower() and 'choice=' in c.lower()]
cond_has_str = pd.Series(False, index=df.index)
cond_dm_str = pd.Series(False, index=df.index)
for c in cols_cronicas:
    is_checked = df[c].astype(str).str.lower().str.strip().isin(['checked', '1', '1.0', 'sim', 'verdadeiro'])
    if 'hipertensão' in c.lower() or 'has' in c.lower():
        cond_has_str = cond_has_str | is_checked
    if 'diabetes' in c.lower() or 'dm' in c.lower():
        cond_dm_str = cond_dm_str | is_checked

df['Tem_HAS'] = (cond_has_text | cond_has_str).astype(int)
df['Tem_DM'] = (cond_dm_text | cond_dm_str).astype(int)

# ------------------------------------------------------------------------------
# 3. CONSOLIDAÇÃO POR PACIENTE ÚNICO (O PADRÃO OURO)
# ------------------------------------------------------------------------------
# Ao agrupar por Record ID e usar o .max(), garantimos 1 paciente = 1 linha
df_pacientes = df.groupby('Record ID')[['Uso_Injetaveis', 'Tem_HAS', 'Tem_DM']].max().reset_index()
df_pacientes['Grupo'] = np.where(df_pacientes['Uso_Injetaveis'] == 1, 'Sim', 'Não')

# ------------------------------------------------------------------------------
# 4. CÁLCULO ESTATÍSTICO (P-VALUE)
# ------------------------------------------------------------------------------
def calcular_pvalue(doenca_coluna):
    tabela_contingencia = pd.crosstab(df_pacientes['Uso_Injetaveis'], df_pacientes[doenca_coluna])
    if tabela_contingencia.shape == (2, 2):
        chi2, p, dof, ex = chi2_contingency(tabela_contingencia, correction=False)
        return round(p, 4)
    return 'N/A'

p_has = calcular_pvalue('Tem_HAS')
p_dm = calcular_pvalue('Tem_DM')

# ------------------------------------------------------------------------------
# 5. CONSTRUÇÃO DA TABELA EXATA DO GRÁFICO
# ------------------------------------------------------------------------------
resultados = []
grupos = ['Sim', 'Não']

for grupo in grupos:
    df_grupo = df_pacientes[df_pacientes['Grupo'] == grupo]
    total_pacientes = len(df_grupo)
    
    if total_pacientes > 0:
        has_abs = df_grupo['Tem_HAS'].sum()
        dm_abs = df_grupo['Tem_DM'].sum()
        
        # Adicionando os dados de Hipertensão
        resultados.append({
            'Comorbidade': 'Hipertensão',
            'Histórico_Injetáveis': grupo,
            'Total_Pacientes_Unicos': total_pacientes,
            'Casos_Confirmados': has_abs,
            'Prevalencia_%': round((has_abs / total_pacientes) * 100, 2),
            'P_Value': p_has
        })
        
        # Adicionando os dados de Diabetes
        resultados.append({
            'Comorbidade': 'Diabetes',
            'Histórico_Injetáveis': grupo,
            'Total_Pacientes_Unicos': total_pacientes,
            'Casos_Confirmados': dm_abs,
            'Prevalencia_%': round((dm_abs / total_pacientes) * 100, 2),
            'P_Value': p_dm
        })

df_tabela = pd.DataFrame(resultados)

# ------------------------------------------------------------------------------
# 6. EXPORTAÇÃO E IMPRESSÃO DA TABELA
# ------------------------------------------------------------------------------
print("\n" + "="*95)
print(f"TABELA DO GRÁFICO: DOENÇAS CRÔNICAS vs INJETÁVEIS (N={len(df_pacientes)} PACIENTES ÚNICOS)")
print("="*95)
print(df_tabela.to_string(index=False))
print("="*95)

nome_arquivo = 'Tabela_Prevalencia_Cronicas_Injetaveis_PACIENTES_UNICOS.csv'
df_tabela.to_csv(nome_arquivo, index=False, encoding='utf-8-sig')
print(f"\n[SUCESSO] A tabela que gera o gráfico foi salva como '{nome_arquivo}'.\n")

# ------------------------------------------------------------------------------
# 7. GERAÇÃO DO GRÁFICO (DESIGN LIMPO SEM BUGS DE SOBREPOSIÇÃO)
# ------------------------------------------------------------------------------
sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 6))

# Cores mantidas do seu estilo visual
cores = ['#d05c53', '#95a5a6']

ax = sns.barplot(
    data=df_tabela,
    x='Comorbidade',
    y='Prevalencia_%',
    hue='Histórico_Injetáveis',
    palette=cores,
    edgecolor='white',
    linewidth=1.5
)

# Textos e Títulos baseados em Pacientes Únicos
total_unicos = len(df_pacientes)
udvi_unicos = len(df_pacientes[df_pacientes['Uso_Injetaveis'] == 1])

plt.title(f'Prevalência de Doenças Crônicas: Usuários de Injetáveis vs População Geral\n(Base: {total_unicos} Pacientes Únicos | {udvi_unicos} UDVI)', 
          fontsize=14, pad=20, fontweight='bold', color='#2c3e50')
plt.ylabel('Prevalência no Grupo (%)', fontsize=12, fontweight='bold', color='#34495e')
plt.xlabel('') 

# Aumenta o limite superior do eixo Y em 20% para que os rótulos não cortem
plt.ylim(0, max(df_tabela['Prevalencia_%']) * 1.25)

# O segredo: ax.bar_label() coloca os rótulos dinamicamente no topo das barras
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=4, fontweight='bold', color='#2c3e50', fontsize=11)

# Remove as bordas do quadrado do gráfico para um visual de artigo/dashboard
sns.despine(left=True)

# Ajusta a Legenda
plt.legend(title='Histórico de Injetáveis', title_fontsize='11', fontsize='10', 
           loc='upper right', frameon=True, shadow=False)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO DOS DADOS E MAPEAMENTO DE COLUNAS
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: O arquivo {file_path} não foi encontrado no diretório atual.")
    raise

# Localizar dinamicamente a coluna do CIAP-2 e de Especialidades
cols_ciap = [c for c in df.columns if 'ciap' in c.lower()]
cols_espec = [c for c in df.columns if 'Para qual especialidade' in c and 'choice=' in c]

# Localizar dinamicamente a coluna de Tempo de Atendimento / Duração
cols_tempo = [c for c in df.columns if 'tempo' in c.lower() or 'duração' in c.lower() or 'duracao' in c.lower()]

# Filtramos apenas os atendimentos reais
df_ev = df[df['Repeat Instrument'].notna()].copy()

# ==============================================================================
# PARTE A: CASCATA CIAP-2 -> ENCAMINHAMENTOS
# ==============================================================================
if cols_ciap and cols_espec:
    col_ciap = cols_ciap[0]

    # Limpeza do CIAP
    df_ciap = df_ev.dropna(subset=[col_ciap]).copy()
    df_ciap = df_ciap[df_ciap[col_ciap].astype(str).str.strip() != '']
    df_ciap = df_ciap[df_ciap[col_ciap].astype(str).str.lower() != 'nan']

    registros_fluxo = []

    # Varremos cada consulta para cruzar CIAP com a Especialidade marcada
    for index, row in df_ciap.iterrows():
        especs_marcadas = [c for c in cols_espec if row[c] == 'Checked']
        ciap_atual = str(row[col_ciap])

        if len(especs_marcadas) == 0:
            destino = 'Sem Encaminhamento (Resolvido)'
            encaminhado = 0
        elif len(especs_marcadas) == 1:
            destino = especs_marcadas[0].split('choice=')[1].replace(')', '').strip()
            encaminhado = 1
        else:
            destino = 'Múltiplos Encaminhamentos'
            encaminhado = 1

        registros_fluxo.append({
            'CIAP': ciap_atual,
            'Destino': destino,
            'Encaminhado': encaminhado
        })

    df_fluxo = pd.DataFrame(registros_fluxo)

    # Otimização do Top 10 para CIAP e Top 10 para Especialidade
    top_ciaps = df_fluxo['CIAP'].value_counts().nlargest(10).index
    top_especs = df_fluxo['Destino'].value_counts().nlargest(10).index

    df_fluxo['CIAP_Plot'] = np.where(df_fluxo['CIAP'].isin(top_ciaps), df_fluxo['CIAP'], 'Outros CIAPs (Agrupados)')
    df_fluxo['Destino_Plot'] = np.where(df_fluxo['Destino'].isin(top_especs), df_fluxo['Destino'], 'Outras Especialidades')

    # Limitação de caracteres para o gráfico não quebrar
    df_fluxo['CIAP_Plot'] = df_fluxo['CIAP_Plot'].apply(lambda x: x[:40] + '...' if len(x) > 40 else x)

    # --- 1. GERAÇÃO DO GRÁFICO (CASCATA) ---
    fig_cascata = px.parallel_categories(
        df_fluxo,
        dimensions=['CIAP_Plot', 'Destino_Plot'],
        color="Encaminhado",
        color_continuous_scale=[(0.0, '#2ecc71'), (1.0, '#e74c3c')], # Verde: Resolvido | Vermelho: Encaminhado
        labels={
            'CIAP_Plot': 'Diagnóstico Inicial (CIAP-2)',
            'Destino_Plot': 'Destino Pós-Consulta'
        },
        title=f"Resolutividade e Encaminhamentos por CIAP-2 (N={len(df_fluxo)} atendimentos)"
    )

    fig_cascata.update_layout(
        height=800,
        font=dict(size=11, family="Arial"),
        coloraxis_showscale=False,
        margin=dict(l=300, r=200, t=100, b=50)
    )
    fig_cascata.show()

    # --- 2. GERAÇÃO E EXPORTAÇÃO DA TABELA (CASCATA) ---
    # Agrupamos as dimensões e contamos o volume de atendimentos em cada fluxo
    df_tabela_fluxo = df_fluxo.groupby(['CIAP_Plot', 'Destino_Plot', 'Encaminhado']).size().reset_index(name='Quantidade_Atendimentos')
    
    # Traduzimos a flag numérica para texto legível
    df_tabela_fluxo['Status_Resolutividade'] = df_tabela_fluxo['Encaminhado'].map({0: 'Resolvido na APS', 1: 'Encaminhado'})
    
    # Ordenamos (Primeiro os resolvidos, depois organizados do maior para o menor volume)
    df_tabela_fluxo = df_tabela_fluxo.sort_values(by=['Encaminhado', 'Quantidade_Atendimentos'], ascending=[True, False])
    df_tabela_fluxo = df_tabela_fluxo[['CIAP_Plot', 'Status_Resolutividade', 'Destino_Plot', 'Quantidade_Atendimentos']].reset_index(drop=True)

    print("\n" + "="*95)
    print("TABELA 1: RESOLUTIVIDADE E FLUXO DE ENCAMINHAMENTOS POR CIAP-2")
    print("="*95)
    print(df_tabela_fluxo.to_string(index=False))
    
    nome_arq_fluxo = 'Tabela_Fluxo_CIAP_Resolutividade.csv'
    df_tabela_fluxo.to_csv(nome_arq_fluxo, index=False, encoding='utf-8-sig')
    print(f"-> Salvo como: {nome_arq_fluxo}\n")

else:
    print("Colunas de CIAP ou Especialidades não encontradas para gerar o gráfico 1.")

# ==============================================================================
# PARTE B: ANÁLISE DE DISTORÇÃO DE TEMPO DE ATENDIMENTO
# ==============================================================================
if cols_tempo:
    col_tempo = cols_tempo[0]

    def extrair_minutos(valor):
        try:
            val_str = str(valor)
            numeros = re.findall(r'\d+', val_str)
            return float(numeros[0]) if numeros else np.nan
        except:
            return np.nan

    df_ev['Tempo_Minutos'] = df_ev[col_tempo].apply(extrair_minutos)
    df_tempo_valido = df_ev[(df_ev['Tempo_Minutos'] >= 1) & (df_ev['Tempo_Minutos'] <= 300)]

    if not df_tempo_valido.empty:
        # --- 1. GERAÇÃO DO GRÁFICO (VIOLINO) ---
        plt.figure(figsize=(12, 6))
        sns.set_theme(style="whitegrid")

        ax = sns.violinplot(x=df_tempo_valido['Tempo_Minutos'], color="#3498db", inner="box")

        mediana = df_tempo_valido['Tempo_Minutos'].median()
        media = df_tempo_valido['Tempo_Minutos'].mean()

        plt.axvline(mediana, color='red', linestyle='--', label=f'Mediana ({mediana:.1f} min)')
        plt.axvline(media, color='green', linestyle=':', label=f'Média ({media:.1f} min)')

        plt.title('Distorção do Tempo de Atendimento (Teleconsultas)', fontsize=15, pad=15, fontweight='bold')
        plt.xlabel(f'Duração da Consulta (minutos) - Coluna: "{col_tempo}"', fontsize=12)
        plt.legend()
        plt.tight_layout()
        plt.show()

        # --- 2. GERAÇÃO E EXPORTAÇÃO DA TABELA (ESTATÍSTICA DE TEMPO) ---
        # Extraímos o resumo estatístico e transformamos num DataFrame limpo
        df_estatisticas = df_tempo_valido['Tempo_Minutos'].describe().round(1).reset_index()
        df_estatisticas.columns = ['Métrica', 'Valor (Minutos)']

        # Traduzimos os nomes das métricas para o português padrão de artigos
        mapa_metricas = {
            'count': 'Total de Consultas Analisadas',
            'mean': 'Tempo Médio',
            'std': 'Desvio Padrão (Variação/Distorção)',
            'min': 'Tempo Mínimo',
            '25%': 'Percentil 25 (1º Quartil)',
            '50%': 'Mediana (Percentil 50)',
            '75%': 'Percentil 75 (3º Quartil)',
            'max': 'Tempo Máximo Absoluto'
        }
        df_estatisticas['Métrica'] = df_estatisticas['Métrica'].map(mapa_metricas)

        print("\n" + "="*80)
        print("TABELA 2: RESUMO ESTATÍSTICO DO TEMPO DE ATENDIMENTO")
        print("="*80)
        print(df_estatisticas.to_string(index=False))
        
        print("\n* INTERPRETAÇÃO PARA GESTÃO:")
        print("A distância entre a Média e a Mediana no gráfico acima, somada ao comprimento do 'pescoço' do violino,")
        print("mostra a quantidade de consultas que fogem da curva normal (Distorção). Casos isolados muito longos")
        print("puxam a média para cima, enquanto a maior parte dos atendimentos se concentra no 'bojo' do gráfico.")
        
        nome_arq_tempo = 'Tabela_Estatistica_Tempo_Atendimento.csv'
        df_estatisticas.to_csv(nome_arq_tempo, index=False, encoding='utf-8-sig')
        print(f"\n-> Salvo como: {nome_arq_tempo}\n")

    else:
        print("A coluna de tempo foi encontrada, mas os dados não puderam ser convertidos para números (minutos).")
else:
    print("\nERRO: Nenhuma coluna contendo 'tempo' ou 'duração' foi encontrada no banco de dados para gerar o gráfico 2.")

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO DOS DADOS E MAPEAMENTO FLEXÍVEL
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: O arquivo {file_path} não foi encontrado no diretório atual.")
    raise

# Isola apenas as consultas realizadas
df_ev = df[df['Repeat Instrument'].notna()].copy()

# Busca a coluna CIAP
cols_ciap = [c for c in df.columns if 'ciap' in c.lower()]
if not cols_ciap:
    raise ValueError("Nenhuma coluna de CIAP encontrada.")
col_ciap = cols_ciap[0]

# Busca as colunas das especialidades de interesse (flexível)
cols_espec = [c for c in df.columns if 'Para qual especialidade' in c and 'choice=' in c]
cols_oftalmo = [c for c in cols_espec if 'oftalmo' in c.lower()]
cols_endocrino = [c for c in cols_espec if 'endocrin' in c.lower()]

# ==============================================================================
# 2. RASTREAMENTO DO FLUXO (CIAP -> OFTALMO/ENDÓCRINO)
# ==============================================================================
# Limpa as linhas sem CIAP
df_ev = df_ev.dropna(subset=[col_ciap])
df_ev = df_ev[df_ev[col_ciap].astype(str).str.strip() != '']
df_ev = df_ev[df_ev[col_ciap].astype(str).str.lower() != 'nan']

registros_fluxo = []

# Varre cada consulta para ver se foi para alguma dessas duas especialidades
for index, row in df_ev.iterrows():
    ciap_atual = str(row[col_ciap])

    # Foi para oftalmologia?
    foi_oftalmo = any(row[c] == 'Checked' for c in cols_oftalmo)
    if foi_oftalmo:
        registros_fluxo.append({
            'CIAP': ciap_atual,
            'Especialidade': 'Oftalmologia',
            'Cor_Fluxo': 0 # 0 = Azul (Oftalmo)
        })

    # Foi para endocrinologia?
    foi_endocrino = any(row[c] == 'Checked' for c in cols_endocrino)
    if foi_endocrino:
        registros_fluxo.append({
            'CIAP': ciap_atual,
            'Especialidade': 'Endocrinologia',
            'Cor_Fluxo': 1 # 1 = Laranja (Endócrino)
        })

df_fluxo = pd.DataFrame(registros_fluxo)

# ==============================================================================
# 3. GERAÇÃO DO DIAGRAMA DE CASCATA (PLOTLY)
# ==============================================================================
if not df_fluxo.empty:
    # Pega os 15 principais CIAPs que geraram esses encaminhamentos
    top_ciaps = df_fluxo['CIAP'].value_counts().nlargest(15).index

    # Agrupa o resto
    df_fluxo['CIAP_Plot'] = np.where(df_fluxo['CIAP'].isin(top_ciaps), df_fluxo['CIAP'], 'Outros CIAPs (Agrupados)')

    # Corta o texto se for muito grande para não quebrar a tela (máximo 45 caracteres)
    df_fluxo['CIAP_Plot'] = df_fluxo['CIAP_Plot'].apply(lambda x: x[:45] + '...' if len(x) > 45 else x)

    # Cria o Gráfico
    fig = px.parallel_categories(
        df_fluxo,
        dimensions=['CIAP_Plot', 'Especialidade'],
        color="Cor_Fluxo",
        color_continuous_scale=[(0.0, '#3498db'), (1.0, '#e67e22')], # Azul (Oftalmo) e Laranja (Endócrino)
        labels={
            'CIAP_Plot': 'Diagnóstico Base (CIAP-2)',
            'Especialidade': 'Especialidade Destino'
        },
        title=f"Fluxo de Encaminhamentos: Oftalmologia vs. Endocrinologia (N={len(df_fluxo)} fluxos)"
    )

    # Ajuste de Layout para caber os nomes
    fig.update_layout(
        height=700,
        font=dict(size=12, family="Arial"),
        coloraxis_showscale=False,
        margin=dict(l=300, r=150, t=100, b=50)
    )

    # Exibe
    fig.show()

    # ==============================================================================
    # 4. EXPORTAÇÃO DA BASE DE DADOS (AGREGADA)
    # ==============================================================================
    # Agrupa para contar exatamente quantos encaminhamentos aconteceram por linha do gráfico
    df_tabela = df_fluxo.groupby(
        ['CIAP_Plot', 'Especialidade']
    ).size().reset_index(name='Quantidade_Encaminhamentos')

    # Ordena para ficar organizado no Excel (por especialidade e depois por volume)
    df_tabela = df_tabela.sort_values(
        by=['Especialidade', 'Quantidade_Encaminhamentos'], 
        ascending=[True, False]
    ).reset_index(drop=True)

    # Salva o arquivo CSV silenciosamente
    nome_arquivo = 'Tabela_Fluxo_Oftalmo_Endocrino.csv'
    df_tabela.to_csv(nome_arquivo, index=False, encoding='utf-8-sig')

    print(f"✅ Sucesso! Gráfico gerado rastreando {len(df_fluxo)} encaminhamentos.")
    print(f"📂 Base de dados salva em: '{nome_arquivo}'.")

else:
    print("Nenhum paciente com CIAP preenchido foi encaminhado para Oftalmologia ou Endocrinologia nesta base.")

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO DOS DADOS E MAPEAMENTO
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: O arquivo {file_path} não foi encontrado no diretório atual.")
    raise

# Isola apenas as consultas (onde os encaminhamentos acontecem)
df_ev = df[df['Repeat Instrument'].notna()].copy()

# Localiza a coluna CIAP-2
cols_ciap = [c for c in df.columns if 'ciap' in c.lower()]
if not cols_ciap:
    raise ValueError("Nenhuma coluna de CIAP encontrada.")
col_ciap = cols_ciap[0]

# Localiza todas as colunas de especialidade
cols_espec = [c for c in df.columns if 'Para qual especialidade' in c and 'choice=' in c]

# ==============================================================================
# 2. RASTREAMENTO GERAL DOS ENCAMINHAMENTOS
# ==============================================================================
# Limpa as linhas onde o médico não preencheu o CIAP
df_ev = df_ev.dropna(subset=[col_ciap])
df_ev = df_ev[df_ev[col_ciap].astype(str).str.strip() != '']
df_ev = df_ev[df_ev[col_ciap].astype(str).str.lower() != 'nan']

registros_totais = []

# Varre o banco cruzando cada CIAP com as especialidades marcadas
for index, row in df_ev.iterrows():
    ciap_atual = str(row[col_ciap])

    for c in cols_espec:
        if row[c] == 'Checked':
            nome_especialidade = c.split('choice=')[-1].replace(')', '').strip()
            registros_totais.append({
                'CIAP': ciap_atual,
                'Especialidade': nome_especialidade
            })

df_fluxo_total = pd.DataFrame(registros_totais)

# ==============================================================================
# 3. FILTRO AUTOMÁTICO: TOP 3 ESPECIALIDADES
# ==============================================================================
if not df_fluxo_total.empty:
    # Descobre quais são as 3 maiores especialidades do sistema
    top_3_especialidades = df_fluxo_total['Especialidade'].value_counts().nlargest(3).index.tolist()

    print("="*60)
    print("🏆 AS 3 ESPECIALIDADES MAIS DEMANDADAS:")
    for i, esp in enumerate(top_3_especialidades, 1):
        print(f" {i}º Lugar: {esp}")
    print("="*60)

    # Filtra o dataframe para manter apenas o fluxo para essas 3 áreas
    df_top3 = df_fluxo_total[df_fluxo_total['Especialidade'].isin(top_3_especialidades)].copy()

    # Mapeia uma cor para cada uma das Top 3 (1, 2 e 3)
    mapeamento_cores = {top_3_especialidades[0]: 0, top_3_especialidades[1]: 0.5, top_3_especialidades[2]: 1}
    df_top3['Cor_Especialidade'] = df_top3['Especialidade'].map(mapeamento_cores)

    # Para o lado esquerdo (CIAP) não virar uma bagunça, pegamos os 15 motivos principais
    # que levam a essas 3 especialidades e agrupamos o resto.
    top_15_ciaps = df_top3['CIAP'].value_counts().nlargest(15).index
    df_top3['CIAP_Plot'] = np.where(df_top3['CIAP'].isin(top_15_ciaps), df_top3['CIAP'], 'Outros CIAPs (Agrupados)')

    # Corta o texto do CIAP em 45 caracteres para não quebrar a tela
    df_top3['CIAP_Plot'] = df_top3['CIAP_Plot'].apply(lambda x: x[:45] + '...' if len(x) > 45 else x)

    # ==============================================================================
    # 4. GERAÇÃO DO DIAGRAMA DE CASCATA (PLOTLY)
    # ==============================================================================
    fig = px.parallel_categories(
        df_top3,
        dimensions=['CIAP_Plot', 'Especialidade'],
        color="Cor_Especialidade",
        color_continuous_scale=[(0.0, '#3498db'), (0.5, '#2ecc71'), (1.0, '#e74c3c')], # Azul, Verde e Vermelho
        labels={
            'CIAP_Plot': 'Diagnóstico Base (CIAP-2)',
            'Especialidade': 'Top 3 Especialidades'
        },
        title=f"Fluxo de Encaminhamentos: CIAP -> Top 3 Especialidades (N={len(df_top3)} fluxos)"
    )

    # Ajuste fino de layout para nomes longos
    fig.update_layout(
        height=750,
        font=dict(size=12, family="Arial"),
        coloraxis_showscale=False,
        margin=dict(l=300, r=150, t=100, b=50) # l=300 cria uma grande margem na esquerda para o texto do CIAP
    )

    # Exibe o gráfico interativo
    fig.show()

    # ==============================================================================
    # 5. EXPORTAÇÃO SILENCIOSA DA BASE DE DADOS (AGREGADA)
    # ==============================================================================
    # Agrupa para contar a espessura exata de cada fluxo visível no gráfico
    df_tabela = df_top3.groupby(
        ['CIAP_Plot', 'Especialidade']
    ).size().reset_index(name='Quantidade_Encaminhamentos')

    # Ordena para ficar bem estruturado no Excel (por especialidade e depois por volume)
    df_tabela = df_tabela.sort_values(
        by=['Especialidade', 'Quantidade_Encaminhamentos'], 
        ascending=[True, False]
    ).reset_index(drop=True)

    # Salva o arquivo CSV com codificação correta para acentos no Excel
    nome_arquivo = 'Tabela_Fluxo_Top3_Especialidades.csv'
    df_tabela.to_csv(nome_arquivo, index=False, encoding='utf-8-sig')

    print(f"\n✅ Sucesso! Gráfico gerado rastreando {len(df_top3)} encaminhamentos para as Top 3 áreas.")
    print(f"📂 Tabela base gerada silenciosamente em: '{nome_arquivo}'.\n")

else:
    print("Não foram encontrados encaminhamentos válidos (com CIAP preenchido) na base de dados.")

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO DOS DADOS E MAPEAMENTO
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: O arquivo {file_path} não foi encontrado.")
    raise

# Isola os eventos de consulta
df_ev = df[df['Repeat Instrument'].notna()].copy()

# Localiza CIAP, Especialidades e Desfecho
cols_ciap = [c for c in df.columns if 'ciap' in c.lower()]
col_ciap = cols_ciap[0] if cols_ciap else None

cols_espec = [c for c in df.columns if 'Para qual especialidade' in c and 'choice=' in c]

# Busca flexível pela coluna de Desfecho/Conduta
cols_desfecho = [c for c in df.columns if any(palavra in c.lower() for palavra in ['desfecho', 'conduta', 'alta', 'resolutividade', 'plano'])]
col_desfecho = cols_desfecho[0] if cols_desfecho else None

if not col_desfecho:
    raise ValueError("Nenhuma coluna de Desfecho ou Conduta encontrada para a IA analisar.")

# ==============================================================================
# 2. ISOLANDO CONSULTAS SEM ENCAMINHAMENTO (RESOLUTIVIDADE APS)
# ==============================================================================
if cols_espec:
    mascara_encaminhado = df_ev[cols_espec].apply(lambda row: (row == 'Checked').any(), axis=1)
    df_aps = df_ev[~mascara_encaminhado].copy()
else:
    df_aps = df_ev.copy()

# Limpa linhas sem desfecho registrado
df_aps = df_aps.dropna(subset=[col_desfecho])
df_aps = df_aps[df_aps[col_desfecho].astype(str).str.strip() != '']

print("="*70)
print(f"🧠 INICIANDO ANÁLISE DE IA NOS DESFECHOS ({len(df_aps)} registros)")
print("="*70)

# ==============================================================================
# 3. MOTOR DE IA (NLP + K-MEANS CLUSTERING)
# ==============================================================================
textos_desfecho = df_aps[col_desfecho].astype(str).tolist()

vectorizer = TfidfVectorizer(
    max_features=150,
    stop_words=None,
    ngram_range=(1, 2)
)

X_vetores = vectorizer.fit_transform(textos_desfecho)

num_clusters = 4
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
df_aps['IA_Cluster_ID'] = kmeans.fit_predict(X_vetores)

termos_ia = vectorizer.get_feature_names_out()
nomes_clusters = {}

for i in range(num_clusters):
    centroide = kmeans.cluster_centers_[i]
    top_indices = centroide.argsort()[-3:][::-1]
    palavras_chave = " + ".join([termos_ia[idx].title() for idx in top_indices])
    # AQUI ESTÁ A MUDANÇA: Tiramos o "Grupo IA:", deixando apenas as chaves
    nomes_clusters[i] = f"[{palavras_chave}]"

df_aps['Desfecho_Agrupado_IA'] = df_aps['IA_Cluster_ID'].map(nomes_clusters)

# ==============================================================================
# 4. SUPERUSUÁRIOS (TOP 10)
# ==============================================================================
contagem_usuarios = df_ev['Record ID'].value_counts()
top_10_ids = contagem_usuarios.head(10).index
df_super = df_aps[df_aps['Record ID'].isin(top_10_ids)].copy()

# ==============================================================================
# 5. GRÁFICO DE DISPERSÃO E EXPORTAÇÃO (LAYOUT COMPACTO)
# ==============================================================================
if col_ciap and not df_super.empty:
    df_super[col_ciap] = df_super[col_ciap].fillna('CIAP Não Informado')
    df_super['CIAP_Plot'] = df_super[col_ciap].astype(str).apply(lambda x: x[:40] + '...' if len(x) > 40 else x)

    df_super['Record ID Str'] = 'Paciente ID: ' + df_super['Record ID'].astype(str)

    df_agg = df_super.groupby(['Record ID Str', 'CIAP_Plot', 'Desfecho_Agrupado_IA']).size().reset_index(name='Volume de Consultas')

    fig = px.scatter(
        df_agg,
        x='Desfecho_Agrupado_IA',
        y='CIAP_Plot',
        size='Volume de Consultas',
        color='Record ID Str',
        hover_name='Record ID Str',
        size_max=35, # Bolhas levemente menores para não sobrepor textos
        title="Dispersão de Superusuários: CIAP vs Conduta (Agrupamento IA)",
        labels={
            'Desfecho_Agrupado_IA': 'Conduta / Desfecho (IA)',
            'CIAP_Plot': 'Diagnóstico (CIAP-2)',
            'Record ID Str': 'Superusuário'
        }
    )

    # AQUI ESTÃO OS AJUSTES DE TAMANHO PARA CABER TUDO
    fig.update_layout(
        height=600, # Gráfico mais achatado (menor altura)
        font=dict(size=11, family="Arial"), # Fonte levemente menor
        margin=dict(l=280, r=50, t=80, b=120), # Margens grandes na Esquerda(l) e Embaixo(b)
        xaxis_tickangle=-15, # Inclinação suave no eixo X para não chocar os nomes
        plot_bgcolor='rgba(240, 240, 240, 0.5)'
    )

    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='white')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='white')

    fig.show()

    # ==============================================================================
    # 6. EXPORTAÇÃO SILENCIOSA DA BASE DE DADOS (AGREGADA)
    # ==============================================================================
    # Ordena a tabela para ficar organizada no Excel (maior volume de consultas no topo)
    df_tabela = df_agg.sort_values(
        by=['Volume de Consultas', 'Record ID Str'], 
        ascending=[False, True]
    ).reset_index(drop=True)

    # Salva o arquivo CSV com codificação correta para acentos no Excel
    nome_arquivo = 'Tabela_Dispersao_Superusuarios_IA.csv'
    df_tabela.to_csv(nome_arquivo, index=False, encoding='utf-8-sig')

    print(f"\n✅ Sucesso! Gráfico de dispersão gerado para os Top 10 superusuários.")
    print(f"📂 Tabela base gerada silenciosamente em: '{nome_arquivo}'.\n")

else:
    print("\n⚠️ Não há dados de CIAP preenchidos para os Superusuários ou os dados são insuficientes.")